# Median DOA Explorer

Tests **Chain B** (inner ring CH5–8, HP 400 Hz, `freq_range=[400, 808]` Hz) with and without the sliding-window circular median.

Run the last cell once per WAV file to compare raw vs median estimates.

In [ ]:
import sys
sys.path.insert(0, '..')

import numpy as np
import matplotlib.pyplot as plt
from logic.wav_loader import WavLoader
from logic.stft_processor import StftProcessor, StftChunk
from logic.doa_processor import DoaProcessor
from logic.median_doa_processor import MedianDoaProcessor
from logic.filters import HighPassFilter


In [ ]:
# --- Config (Chain B) ---
SAMPLE_RATE  = 44100
CHUNK_SIZE   = 8192
NPERSEG      = 512
NOVERLAP     = 256
INNER_IDX    = slice(4, 8)   # CH5-8
HP_CUTOFF_HZ = 400
FREQ_RANGE   = [400, 808]
MEDIAN_WINDOW = 3

L_inner = np.array([
    [-0.075,  0.075, -0.075,  0.075],
    [-0.075, -0.075,  0.075,  0.075],
    [ 0.000,  0.000,  0.000,  0.000],
])


In [ ]:
def run_pipeline(wav_file):
    """Returns (timestamps, raw_degs, median_degs) for the given WAV file."""

    def _stft_stream(wav_file):
        hp   = HighPassFilter(cutoff_hz=HP_CUTOFF_HZ, sampling_rate=SAMPLE_RATE, order=4)
        stft = StftProcessor(nperseg=NPERSEG, noverlap=NOVERLAP)
        for c in stft.process(hp.process(WavLoader(wav_file, CHUNK_SIZE).stream())):
            yield StftChunk(c.freqs, c.times, c.magnitudes[INNER_IDX],
                            c.sampling_rate, c.timestamp)

    # Raw DOA
    hp1   = HighPassFilter(cutoff_hz=HP_CUTOFF_HZ, sampling_rate=SAMPLE_RATE, order=4)
    stft1 = StftProcessor(nperseg=NPERSEG, noverlap=NOVERLAP)
    doa1  = DoaProcessor(mic_locs=L_inner, sampling_rate=SAMPLE_RATE,
                          nfft=NPERSEG, freq_range=FREQ_RANGE)
    raw_chunks = list(doa1.process(_stft_stream(wav_file)))
    raw_t   = [c.timestamp   for c in raw_chunks]
    raw_deg = [c.azimuth_deg for c in raw_chunks]

    # Median DOA (re-run pipeline — generators are one-shot)
    doa2 = DoaProcessor(mic_locs=L_inner, sampling_rate=SAMPLE_RATE,
                         nfft=NPERSEG, freq_range=FREQ_RANGE)
    med  = MedianDoaProcessor(window=MEDIAN_WINDOW)
    med_chunks = list(med.process(doa2.process(_stft_stream(wav_file))))
    med_deg = [c.azimuth_deg for c in med_chunks]

    return raw_t, raw_deg, med_deg


In [ ]:
# ── Change this to test different files ─────────────────────────────────────
WAV_FILE       = '../data/static_10m_000.wav'
EXPECTED_DEG   = 0       # ground truth for annotation
# ─────────────────────────────────────────────────────────────────────────────

timestamps, raw_deg, med_deg = run_pipeline(WAV_FILE)

# --- Print table ---
SEP = '─' * 52
print(SEP)
print(f"{'time':>8}  {'raw DOA':>10}  {'median DOA':>12}  {'err (med)':>10}")
print(SEP)
for t, r, m in zip(timestamps, raw_deg, med_deg):
    err = min(abs(m - EXPECTED_DEG), 360 - abs(m - EXPECTED_DEG))
    print(f"{t:7.3f}s  {r:9.1f}°  {m:11.1f}°  {err:9.1f}°")
print(SEP)

raw_errs = [min(abs(r-EXPECTED_DEG), 360-abs(r-EXPECTED_DEG)) for r in raw_deg]
med_errs = [min(abs(m-EXPECTED_DEG), 360-abs(m-EXPECTED_DEG)) for m in med_deg]
print(f"Mean error — raw: {np.mean(raw_errs):.1f}°   median: {np.mean(med_errs):.1f}°")

# --- Plot ---
fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(timestamps, raw_deg, 'o--', color='steelblue', label='Raw DOA', alpha=0.7)
ax.plot(timestamps, med_deg, 's-',  color='tomato',    label=f'Median (window={MEDIAN_WINDOW})', linewidth=2)
ax.axhline(EXPECTED_DEG, color='green', linestyle=':', linewidth=1.5, label=f'Ground truth {EXPECTED_DEG}°')
ax.set_xlabel('Time (s)')
ax.set_ylabel('Azimuth (°)')
ax.set_title(f'Raw vs Median DOA — {WAV_FILE.split("/")[-1]}')
ax.set_ylim(-5, 365)
ax.set_yticks(range(0, 361, 45))
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()
